In [ ]:
# autoreload magic
%load_ext autoreload
%autoreload 2

In [ ]:
from lisatools.globalfit.preprocessing import L1ProcessingStep
from lisatools.detector import L1Orbits
from lisatools.domains import STFTSettings
from lisatools.utils.utility import tukey
from lisatools.utils.constants import YRSID_SI

In [ ]:
mojito_path = "/Volumes/as_storage/mojito/mojito_light_2p5s"

In [ ]:
loader = L1ProcessingStep(
        L1_folder=mojito_path,
        source_types=["noise"],
        orbits_class=L1Orbits,
        orbits_kwargs=dict(force_backend="cpu", frame="icrs"),
        verbose=True,
    )

In [ ]:
dt = 5
downsample_kwargs = {
    "target_fs": 1 / dt,
    "window": ("kaiser", 5.0), 
}
preprocess_kwargs = dict(
        highpass_kwargs=dict(cutoff=5e-6, order=2, zero_phase=True),
        lowpass_kwargs=dict(cutoff=0.8 * downsample_kwargs['target_fs'], order=2, zero_phase=True),
        trim_kwargs=dict(
            duration=200 * 3600, is_percent=False, trimming_type="from_each_end"
        ),
        downsample_kwargs=downsample_kwargs,
        Tobs=2. * YRSID_SI
)

In [ ]:
loader.process(**preprocess_kwargs)

In [ ]:
DT = 7 * 24 * 3600  # day in seconds
TAPER = 600 * 2 # seconds
settings_factory = STFTSettings.make_factory(big_dt=DT, min_freq=1e-4)#, max_freq=1e-1)
settings = settings_factory(loader.times, loader.dt, 'cpu')

nperseg = settings.get_nperseg(loader.dt)
window = tukey(nperseg, alpha=TAPER / DT)
print(f"window length: {nperseg} samples, {nperseg * loader.dt:.2f} seconds")
print(f"window taper: {TAPER} seconds")
print(f"window parameter: {TAPER / DT}")

In [ ]:
data, orbits = loader.pour(settings=settings, window=window, return_orbits=True)

In [ ]:
response = loader.get_total_response(data.settings.f_arr)

In [ ]:
loader.original_fs / loader.fs

In [ ]:
import matplotlib.pyplot as plt
plt.loglog(data.settings.f_arr, response)


In [ ]:
def get_window_kernel(window_values):
    """Two-sided spectral window |W[k]|^2 / N^2 on the full DFT circle (length N)."""
    return np.abs(np.fft.fft(window_values)) ** 2 / len(window_values) ** 2

def convolve_matrix_components(entries, window_values, active_slice=slice(None)):
    """Convolve one-sided PSD entries with the window's two-sided spectral window.

    ``entries`` may live on the active band only; they are embedded into the full
    one-sided grid (out-of-band PSD assumed zero), convolved on the full length-N
    circle, and sliced back. Pass the full grid whenever you can.
    """
    N = len(window_values)
    nf_full = N // 2 + 1
    entries = np.asarray(entries)
    fill_low = entries[..., 0:1]
    fill_high = entries[..., -1:]
    full = np.ones(entries.shape[:-1] + (nf_full,), dtype=complex) * fill_low
    full[..., active_slice] = entries
    #  pad the low-frequency entries with the lowest frequency (DC) value
    full[..., active_slice.stop:] = fill_high

    two_sided = np.concatenate([full, np.conj(full[..., 1:(N + 1) // 2][..., ::-1])], axis=-1)
    kernel = np.abs(np.fft.fft(window_values)) ** 2 / N ** 2
    conv = np.fft.ifft(np.fft.fft(two_sided, axis=-1) * np.fft.fft(kernel), axis=-1)
    conv = conv[..., :nf_full][..., active_slice]
    return [np.real(conv[i]) for i in range(3)] + [conv[i] for i in range(3, 6)]


In [ ]:
from lisatools.sensitivity import XYZSensitivityBackend, SensitivityMatrixBase
import numpy as np

In [ ]:
sens_backend = XYZSensitivityBackend(orbits, settings=settings, window_values=window)
sens_backend_no_norm = XYZSensitivityBackend(orbits, settings=settings, window_values=None)
sens_backend_convolve = XYZSensitivityBackend(orbits, settings=settings, window_values=window, convolve_window=True, filters_response=response, mask_percentage=0.0, smoothing_sigma=0)

In [ ]:
from lisatools.analysiscontainer import AnalysisContainer

In [ ]:
assert isinstance(sens_backend, SensitivityMatrixBase)

In [ ]:
sens_backend_convolve.set_sensitivity_matrix()

ac = AnalysisContainer(
    data=data, sens_mat=sens_backend_convolve
)

In [ ]:
print(sens_backend.noise_normalization)
print(sens_backend_no_norm.noise_normalization)
print(sens_backend_convolve.noise_normalization)

In [ ]:
sens_mat_smooth = sens_backend.compute_sensitivity_matrix( smooth=True)
sens_mat = sens_backend.compute_sensitivity_matrix( smooth=False)
sens_mat_no_norm = sens_backend_no_norm.compute_sensitivity_matrix( smooth=False)

In [ ]:
sens_mat_convolved = sens_backend_convolve.compute_sensitivity_matrix( smooth=False)

In [ ]:
tf_periodogram = np.einsum('itf,jtf->ijtf', data.arr, np.conj(data.arr)) * 2 * settings.df

In [ ]:
from cantuccio.visuals import get_paper_style

In [ ]:
style = get_paper_style()
plt.style.use(style)

In [ ]:
settings.f_arr

In [ ]:
get_window_kernel(window)

In [ ]:
np.fft.fft(window)

In [ ]:
plt.figure()

ch0, ch1 = 0, 0
time_bin = 0
mask = sens_backend_convolve.dips_mask.reshape(sens_backend_convolve.basis_settings.basis_shape)
dips_frequencies = settings.f_arr[mask[time_bin]]

plt.loglog(data.settings.f_arr, np.abs(tf_periodogram[ch0, ch1, time_bin, :]), label="X", c='lightgrey')
plt.loglog(data.settings.f_arr, np.abs(sens_mat_no_norm[ch0, ch1, time_bin, :]), label="unnormalized sensitivity", c='k')
plt.loglog(data.settings.f_arr, np.abs(sens_mat[ch0, ch1, time_bin, :]), label="X sensitivity", c='blue')
plt.loglog(data.settings.f_arr, np.abs(sens_mat[ch0, ch1, time_bin, :] * response), label="filtered sensitivity", c='red', ls='--')
plt.loglog(data.settings.f_arr, np.abs(sens_mat_smooth[ch0, ch1, time_bin, :] * response), label="smoothed filtered sensitivity", c='green', ls='-.')
plt.loglog(data.settings.f_arr, np.abs(sens_mat_convolved[ch0, ch1, time_bin, :]), label="convolved filtered sensitivity", c='purple', ls=':')
plt.loglog(data.settings.f_arr, np.abs(ac.sens_mat.sens_mat[ch0, ch1, time_bin, :]), label="convolved filtered sensitivity", c='k', ls='--')

# plot the dips mask
plt.scatter(dips_frequencies, np.ones_like(dips_frequencies) * 1e-40, marker='o', color='red', alpha=0.01, label="dips mask", zorder=10)

# plot the vertical line matching the roll-off frequency of the window

plt.axvline(x=1/(2*TAPER), color='orange', ls='--', label="window roll-off frequency")

plt.xlim(1e-4, 1e-1)
#plt.xlim(2.95e-2, 3.05e-2)
plt.xlim(8.5e-2, 1e-1)
plt.ylim(1e-44, 1e-37)
plt.show()

In [ ]:
dips_frequencies

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3))

ranges = [(1e-4, 1e-2), (1e-2, 1e-1)]

for i, (ax, freq_range) in enumerate(zip(axs, ranges)):
    mask = (data.settings.f_arr >= freq_range[0]) & (data.settings.f_arr <= freq_range[1])
    ax.semilogx(data.settings.f_arr[mask], (sens_mat_convolved[ch0, ch1, time_bin, :] / sens_mat[ch0, ch1, time_bin, :])[mask], label="convolved / normalized", c='k', ls='-')
    ax.semilogx(data.settings.f_arr[mask], (sens_mat_convolved[ch0, ch1, time_bin, :] / sens_mat_no_norm[ch0, ch1, time_bin, :])[mask], label="convolved / unnormalized", c='k', ls='--')
    ax.semilogx(data.settings.f_arr[mask], (sens_mat[ch0, ch1, time_bin, :] / sens_mat_no_norm[ch0, ch1, time_bin, :])[mask], label="normalized / unnormalized", c='k', ls=':')

    if 1/(2*TAPER) >= freq_range[0] and 1/(2*TAPER) <= freq_range[1]:
        f_taper = 1/(2*TAPER)
        ax.axvline(x=f_taper, color='k', lw=1, ls='--', label="window roll-off frequency")

    ax.legend()
    ax.set_xlabel('Frequency (Hz)')

axs[0].set_ylabel('Ratio')
axs[1].set_yscale('log')

## check the likelihood

In [ ]:
NUM_WALKERS = 2

S_oms = np.array([15e-12] * NUM_WALKERS) * np.array([1.0, 0.9])
S_acc = np.array([3e-15] * NUM_WALKERS) * np.array([1.0, 0.9])
Amp = alpha = f_1 = f_kn = f_2 = np.array([0.0] * NUM_WALKERS)
data_index = np.arange(NUM_WALKERS, dtype=np.int32)

data_ll = np.array([data.arr.copy() for _ in range(NUM_WALKERS)])

In [ ]:
sens_backend_convolve.compute_log_like(data_ll, data_index, S_oms, S_acc, Amp, alpha, f_1, f_kn, f_2)

In [ ]:
sens_backend_convolve.compute_log_like(data_ll, data_index, S_oms, S_acc, Amp, alpha, f_1, f_kn, f_2, convolve_window=False)

In [ ]:
(ac.likelihood() - 1.62263095e+09) / 1e9

In [ ]:
from lisatools.analysiscontainer import AnalysisContainer

In [ ]:
ac = AnalysisContainer(data=data, sens_mat=sens_backend_convolve)

In [ ]:
import lisatools
isinstance(sens_backend, lisatools.sensitivity.SensitivityMatrixBase)

### Frequency-frequency matrix of the data_channel

In [ ]:
channel = 0

# One segment's outer product d d^H is rank-1: |d_k||d_k'| everywhere, so it has no
# diagonal structure by construction. The frequency-frequency covariance is an
# expectation, so average the outer product over the STFT time segments.
d = data.arr[channel]          # (NT, NF)
NT, NF = d.shape
print(f"{NT} segments x {NF} frequency bins")

In [ ]:
# Banded estimate of C[k, k+q] = <d_k conj(d_{k+q})>_t for lags q = 0..B (full NF x NF is 1.2 GB
# and everything beyond a few tens of bins is noise).
B = 500
C_band = np.zeros((B + 1, NF), dtype=complex)
for q in range(B + 1):
    C_band[q, : NF - q] = np.mean(d[:, : NF - q] * np.conj(d[:, q:]), axis=0)

# correlation coefficient per lag. C[k,k+q] ~ S[k] conj(V[q]) / N has the same phase at every k
# (smooth PSD), so average the normalised complex value over bins coherently: the 1/sqrt(NT)
# estimation noise then averages down by a further 1/sqrt(n_bins) instead of adding in quadrature.
f = data.settings.f_arr
smooth = (f > 1e-3) & (f < 2e-2)          # away from DC and the TDI notches
C0 = np.real(C_band[0])
rho_meas = np.array([
    np.abs(np.mean((C_band[q, : NF - q] / np.sqrt(C0[: NF - q] * C0[q:]))[smooth[: NF - q]]))
    for q in range(B + 1)
])

# prediction from the window alone (smooth PSD): C[k, k+q] ~ S[k] V[q] / N with V = fft(w^2)
V = np.fft.fft(window ** 2)
rho_pred = np.abs(V[: B + 1]) / V[0].real
noise_floor = 1 / np.sqrt(NT * smooth.sum())   # coherent-average noise floor on rho_meas
elem_floor = 1 / np.sqrt(NT)                    # noise on a single matrix element (block image below)
print("lag  rho_meas  rho_pred")
for q in range(6):
    print(f"{q:3d}   {rho_meas[q]:.4f}    {rho_pred[q]:.4f}")
print(f"noise floor: {noise_floor:.4f} on rho_meas, {elem_floor:.3f} per matrix element")

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5))

# |rho| vs lag: measured against the window prediction
lags = np.arange(B + 1)
ax0.semilogy(lags, rho_meas, "o", label="measured  |C[k,k+q]| / sqrt(C[k,k] C[k+q,k+q])")
ax0.semilogy(lags, rho_pred, "-", label="window prediction  |fft($w^2$)[q]| / sum($w^2$)")
ax0.axhline(noise_floor, ls="--", c="grey", label="noise floor of the estimate")
ax0.set_xlabel("lag q (bins)"); ax0.set_ylabel(r"$|\rho_q|$"); ax0.set_ylim(1e-3, 2); ax0.legend(fontsize=8)

# a zoomed block of the normalised covariance, linear frequency axes
k0 = int(3e-3 / data.settings.df); nb = B
block = np.mean(d[:, k0 : k0 + nb, None] * np.conj(d[:, None, k0 : k0 + nb]), axis=0)
diag = np.real(np.diag(block))
corr = np.abs(block) / np.sqrt(np.outer(diag, diag))
ext = [f[k0], f[k0 + nb - 1], f[k0], f[k0 + nb - 1]]
from matplotlib.colors import LogNorm
im = ax1.imshow(corr, origin="lower", extent=ext, norm=LogNorm(vmin=1e-2, vmax=1), cmap="cividis")
ax1.set_xlabel("Frequency (Hz)"); ax1.set_ylabel("Frequency (Hz)")
ax1.set_title(f"|corr| block, {nb} bins at {f[k0]:.2e} Hz  (single-element noise ~{elem_floor:.2f})")
fig.colorbar(im, ax=ax1)

plt.show()